# RAG Basics: Loading PDF Documents

This notebook loads the tutorial arXiv papers with LangChain's `PyPDFLoader`. Each PDF page becomes a LangChain `Document` containing page text and source metadata.

In [1]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader


def find_project_root(start: Path) -> Path:
    """Find the nearest parent directory containing pyproject.toml."""
    for directory in (start, *start.parents):
        if (directory / "pyproject.toml").exists():
            return directory
    raise FileNotFoundError("Could not locate the project root")


project_root = find_project_root(Path.cwd().resolve())
pdf_directory = project_root / "data" / "raw" / "arxiv"
pdf_paths = sorted(pdf_directory.glob("*.pdf"))

if not pdf_paths:
    raise FileNotFoundError(
        f"No PDFs found in {pdf_directory}. Run "
        "`uv run python -m src.download_arxiv_papers` first."
    )

documents = []
for pdf_path in pdf_paths:
    loader = PyPDFLoader(pdf_path)
    documents.extend(loader.load())

print(f"Loaded {len(pdf_paths)} PDF files from {pdf_directory}")
print(f"Created {len(documents)} page-level documents")
print("Files:")
for pdf_path in pdf_paths:
    page_count = sum(
        document.metadata.get("source") == str(pdf_path)
        for document in documents
    )
    print(f"- {pdf_path.name}: {page_count} pages")

first_document = documents[0]
print("\nFirst document metadata:")
print(first_document.metadata)
print("\nFirst 500 characters:")
print(first_document.page_content[:500])

/var/folders/hf/0h4pd6zx1vv0dwjx8bgs9r_c0000gp/T/ipykernel_91708/609406098.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 5 PDF files from /Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/data/raw/arxiv
Created 103 page-level documents
Files:
- 2606.20331.pdf: 32 pages
- 2606.20374.pdf: 17 pages
- 2606.20497.pdf: 27 pages
- 2606.20539.pdf: 6 pages
- 2606.20563.pdf: 21 pages

First document metadata:
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Robert Ganian; Mathis Rocton', 'doi': 'https://doi.org/10.48550/arXiv.2606.20331', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Computing Twin-Width via Treedepth and Vertex Integrity', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2606.20331v1', 'source': '/Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/data/raw/arxiv/2606.20331.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}

First 500 characters:
Computing Twin-Width
via Treedept

## Split pages into retrieval chunks

Embedding complete pages often produces vectors that represent too many ideas at once. `RecursiveCharacterTextSplitter` creates smaller, overlapping chunks while preferentially splitting on paragraph, line, and word boundaries.

For this baseline, we use `chunk_size=1000` and `chunk_overlap=200`. These are practical starting values for academic text: chunks retain enough context for question answering, while the overlap reduces information loss at chunk boundaries. They should later be tuned using retrieval evaluation rather than treated as universal defaults.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(documents)
chunk_lengths = [len(chunk.page_content) for chunk in chunks]

print(f"Split {len(documents)} page documents into {len(chunks)} chunks")
print(f"Average chunk length: {sum(chunk_lengths) / len(chunk_lengths):.0f} characters")
print(f"Chunk length range: {min(chunk_lengths)}-{max(chunk_lengths)} characters")

sample_chunk = chunks[0]
print("\nSample chunk metadata:")
print(sample_chunk.metadata)
print("\nSample chunk text:")
print(sample_chunk.page_content[:1000])

Split 103 page documents into 445 chunks
Average chunk length: 869 characters
Chunk length range: 72-1000 characters

Sample chunk metadata:
{'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Robert Ganian; Mathis Rocton', 'doi': 'https://doi.org/10.48550/arXiv.2606.20331', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'Computing Twin-Width via Treedepth and Vertex Integrity', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2606.20331v1', 'source': '/Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/data/raw/arxiv/2606.20331.pdf', 'total_pages': 32, 'page': 0, 'page_label': '1'}

Sample chunk text:
Computing Twin-Width
via Treedepth and Vertex Integrity
Robert Ganian/envel⌢pe/orcid
Algorithms and Complexity Group, TU Wien, Vienna, Austria
Mathis Rocton/envel⌢pe/orcid
Algorithms and Comple

## Choose an embedding model

The default provider is Hugging Face, which runs locally and requires no API key. `sentence-transformers/all-MiniLM-L6-v2` is a lightweight semantic-search model that produces 384-dimensional vectors. Set `EMBEDDING_PROVIDER` to `"openai"` to use `text-embedding-3-small` instead.

The same embedding model must be used both when indexing document chunks and when embedding retrieval queries. Changing the provider or model requires rebuilding the vector database.

In [3]:
from dotenv import load_dotenv


EMBEDDING_PROVIDER = "huggingface"  # Options: "huggingface", "openai"
HF_EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

if EMBEDDING_PROVIDER == "huggingface":
    from langchain_huggingface import HuggingFaceEmbeddings

    embedding_model_name = HF_EMBEDDING_MODEL
    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model_name,
        model_kwargs={"device": "cpu"},
        encode_kwargs={"normalize_embeddings": True},
    )
elif EMBEDDING_PROVIDER == "openai":
    from langchain_openai import OpenAIEmbeddings

    load_dotenv(project_root / ".env")
    embedding_model_name = OPENAI_EMBEDDING_MODEL
    embeddings = OpenAIEmbeddings(model=embedding_model_name)
else:
    raise ValueError(
        "EMBEDDING_PROVIDER must be either 'huggingface' or 'openai'"
    )

sample_embedding = embeddings.embed_query(
    "How do the papers approach their main research problems?"
)

print(f"Embedding provider: {EMBEDDING_PROVIDER}")
print(f"Embedding model: {embedding_model_name}")
print(f"Embedding dimensions: {len(sample_embedding)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14303.18it/s]

Embedding provider: huggingface
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding dimensions: 384


## Build a persistent Chroma vector database

Each embedding model gets its own storage directory because vector dimensions and representations are not interchangeable. The database configuration is recorded in `manifest.json`, and a content fingerprint prevents an incompatible index from being reused after documents, chunking parameters, or embedding settings change.

Set `REBUILD_VECTOR_DB = True` when you intentionally want to replace the selected provider's index.

In [4]:
import hashlib
import json
import re
import shutil

from langchain_chroma import Chroma


REBUILD_VECTOR_DB = False
COLLECTION_NAME = "arxiv-rag-basics"


def safe_directory_name(value: str) -> str:
    """Convert a provider/model name into a filesystem-safe name."""
    return re.sub(r"[^a-zA-Z0-9._-]+", "-", value).strip("-").lower()


def chunk_id(chunk: object, index: int) -> str:
    """Create a stable ID from the chunk's source, page, position, and text."""
    metadata = chunk.metadata
    identity = "|".join(
        [
            str(metadata.get("source", "")),
            str(metadata.get("page", "")),
            str(index),
            chunk.page_content,
        ]
    )
    return hashlib.sha256(identity.encode("utf-8")).hexdigest()


chunk_ids = [chunk_id(chunk, index) for index, chunk in enumerate(chunks)]
content_fingerprint = hashlib.sha256(
    "".join(chunk_ids).encode("utf-8")
).hexdigest()

index_name = safe_directory_name(
    f"{EMBEDDING_PROVIDER}-{embedding_model_name}"
)
vector_db_directory = (
    project_root / "data" / "processed" / "chroma" / index_name
)
manifest_path = vector_db_directory / "manifest.json"
index_config = {
    "collection_name": COLLECTION_NAME,
    "embedding_provider": EMBEDDING_PROVIDER,
    "embedding_model": embedding_model_name,
    "embedding_dimensions": len(sample_embedding),
    "chunk_size": 1000,
    "chunk_overlap": 200,
    "chunk_count": len(chunks),
    "content_fingerprint": content_fingerprint,
}

if REBUILD_VECTOR_DB and vector_db_directory.exists():
    shutil.rmtree(vector_db_directory)

if manifest_path.exists():
    existing_config = json.loads(manifest_path.read_text())
    if existing_config != index_config:
        raise RuntimeError(
            "The existing vector database configuration does not match "
            "the current documents or embedding settings. Set "
            "REBUILD_VECTOR_DB = True and rerun this cell."
        )

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(vector_db_directory),
    collection_metadata={"hnsw:space": "cosine"},
)

existing_count = vector_store._collection.count()
if existing_count == 0:
    vector_store.add_documents(documents=chunks, ids=chunk_ids)
    vector_db_directory.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text(json.dumps(index_config, indent=2) + "\n")
    action = "Created"
elif existing_count == len(chunks):
    action = "Loaded"
else:
    raise RuntimeError(
        f"Expected {len(chunks)} vectors but found {existing_count}. "
        "Set REBUILD_VECTOR_DB = True and rerun this cell."
    )

stored_count = vector_store._collection.count()
print(f"{action} vector database: {vector_db_directory}")
print(f"Collection: {COLLECTION_NAME}")
print(f"Embedding model: {embedding_model_name}")
print(f"Stored vectors: {stored_count}")

if stored_count != len(chunks):
    raise RuntimeError(
        f"Vector count mismatch: expected {len(chunks)}, found {stored_count}"
    )

Loaded vector database: /Users/mlstudio/Claude-Code/ml-lab-new/tutorial-agentic-ai/data/processed/chroma/huggingface-sentence-transformers-all-minilm-l6-v2
Collection: arxiv-rag-basics
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Stored vectors: 445


## Test similarity retrieval

The cases below select one distinctive passage from each paper and search for it semantically. A retrieval passes when a chunk from the expected PDF and page appears in the top five results. We also report its rank because ranking quality matters even when the correct passage is retrieved.

These checks are a small retrieval smoke test, not a full evaluation dataset. Different embedding models may rank the same relevant chunks differently.

In [5]:
from pathlib import Path


retrieval_cases = [
    {
        "expected_file": "2606.20331.pdf",
        "expected_page": 2,
        "query": (
            "Whether twin-width admits any fixed-parameter approximation "
            "is arguably the central open problem in the area."
        ),
    },
    {
        "expected_file": "2606.20374.pdf",
        "expected_page": 2,
        "query": (
            "The training execution hierarchy includes CPU call-stack "
            "profiling, Python scheduling, framework semantics, and GPU "
            "kernel tracing."
        ),
    },
    {
        "expected_file": "2606.20497.pdf",
        "expected_page": 3,
        "query": (
            "The core surrogate is a linear graphlet model trained with "
            "a meta-learning formulation for multi-objective molecular "
            "discovery."
        ),
    },
    {
        "expected_file": "2606.20539.pdf",
        "expected_page": 2,
        "query": (
            "Keeping the large cold 1GB object saves about $0.90, over "
            "four orders of magnitude more dollars for fewer hits."
        ),
    },
    {
        "expected_file": "2606.20563.pdf",
        "expected_page": 3,
        "query": (
            "A zero-shot two-stage framework generates coherent 3D visual "
            "illusions in three to five minutes."
        ),
    },
]

top_k = 5
retrieval_results = []

for case in retrieval_cases:
    matches = vector_store.similarity_search_with_score(
        case["query"],
        k=top_k,
    )
    expected_rank = None

    for rank, (document, score) in enumerate(matches, start=1):
        source_file = Path(document.metadata["source"]).name
        page_number = int(document.metadata["page"]) + 1
        if (
            source_file == case["expected_file"]
            and page_number == case["expected_page"]
        ):
            expected_rank = rank
            break

    top_document, top_score = matches[0]
    retrieval_results.append(
        {
            "expected": (
                f"{case['expected_file']} p.{case['expected_page']}"
            ),
            "top_result": (
                f"{Path(top_document.metadata['source']).name} "
                f"p.{int(top_document.metadata['page']) + 1}"
            ),
            "expected_rank": expected_rank,
            "top_score": top_score,
            "passed": expected_rank is not None,
        }
    )

print(f"Retrieval test using {embedding_model_name}\n")
for index, result in enumerate(retrieval_results, start=1):
    status = "PASS" if result["passed"] else "FAIL"
    rank = result["expected_rank"] or "not found"
    print(
        f"{index}. {status} | expected {result['expected']} | "
        f"top result {result['top_result']} | expected rank {rank} | "
        f"top distance {result['top_score']:.4f}"
    )

passed_count = sum(result["passed"] for result in retrieval_results)
print(f"\nPassed {passed_count}/{len(retrieval_results)} retrieval cases")

if passed_count != len(retrieval_results):
    raise AssertionError(
        "At least one expected source/page was absent from the top "
        f"{top_k} results"
    )

Retrieval test using sentence-transformers/all-MiniLM-L6-v2

1. PASS | expected 2606.20331.pdf p.2 | top result 2606.20331.pdf p.2 | expected rank 1 | top distance 0.3317
2. PASS | expected 2606.20374.pdf p.2 | top result 2606.20374.pdf p.2 | expected rank 1 | top distance 0.2985
3. PASS | expected 2606.20497.pdf p.3 | top result 2606.20497.pdf p.3 | expected rank 1 | top distance 0.2178
4. PASS | expected 2606.20539.pdf p.2 | top result 2606.20539.pdf p.2 | expected rank 1 | top distance 0.4130
5. PASS | expected 2606.20563.pdf p.3 | top result 2606.20563.pdf p.2 | expected rank 2 | top distance 0.3207

Passed 5/5 retrieval cases


## Build the RAG chain with ChatGroq

The basic chain retrieves the four most similar chunks, formats them with source and page labels, and asks `qwen/qwen3-32b` to answer only from that context. The prompt requires inline citations and an explicit insufficient-context response instead of unsupported guesses.

The chain returns both the generated answer and the retrieved `Document` objects so retrieval and generation remain independently inspectable.

In [6]:
from pathlib import Path

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_groq import ChatGroq


GROQ_MODEL = "qwen/qwen3-32b"
RETRIEVAL_K = 4

load_dotenv(project_root / ".env")

retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": RETRIEVAL_K},
)


def format_retrieved_documents(retrieved_documents: list) -> str:
    """Format retrieved chunks with source labels for grounded citations."""
    formatted_documents = []
    for index, document in enumerate(retrieved_documents, start=1):
        source_file = Path(document.metadata["source"]).name
        page_number = int(document.metadata["page"]) + 1
        formatted_documents.append(
            f"[Source {index}: {source_file}, page {page_number}]\n"
            f"{document.page_content}"
        )
    return "\n\n".join(formatted_documents)


rag_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You answer questions about the supplied research papers. "
            "Use only the retrieved context. If the context does not "
            "contain enough information, say that you do not have enough "
            "information. Cite factual claims inline using the exact "
            "source filename and page number, for example "
            "[2606.20331.pdf, page 3]. Do not cite sources that do not "
            "support the claim. Answer concisely in two to four short "
            "paragraphs.\n\nRetrieved context:\n{context}",
        ),
        ("human", "{question}"),
    ]
)

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    max_tokens=2048,
    max_retries=2,
    reasoning_format="hidden",
)

retrieval_chain = RunnablePassthrough.assign(
    documents=RunnableLambda(
        lambda inputs: retriever.invoke(inputs["question"])
    )
)

answer_chain = (
    RunnablePassthrough.assign(
        context=RunnableLambda(
            lambda inputs: format_retrieved_documents(inputs["documents"])
        )
    )
    | RunnablePassthrough.assign(
        answer=rag_prompt | llm | StrOutputParser()
    )
)

rag_chain = retrieval_chain | answer_chain

print(f"RAG model: {GROQ_MODEL}")
print(f"Retriever: similarity search with k={RETRIEVAL_K}")

RAG model: qwen/qwen3-32b
Retriever: similarity search with k=4


## Ask a grounded question

This example asks about one paper and displays both the answer and the chunks supplied to the model.

In [7]:
question = (
    "Why can optimizing cache hit rate be financially suboptimal for "
    "cloud object storage?"
)

result = rag_chain.invoke({"question": question})

print(f"Question: {question}\n")
print("Answer:")
print(result["answer"])
print("\nRetrieved sources:")
for document in result["documents"]:
    source_file = Path(document.metadata["source"]).name
    page_number = int(document.metadata["page"]) + 1
    print(f"- {source_file}, page {page_number}")

Question: Why can optimizing cache hit rate be financially suboptimal for cloud object storage?

Answer:
Optimizing cache hit rate can be financially suboptimal for cloud object storage because cloud billing prioritizes minimizing total cost (per GET request and per byte of egress) over reducing latency or misses. Classic caching strategies like LRU focus on maximizing hit rates, but this ignores heterogeneous miss costs: a large, infrequently accessed object with high egress fees might cost orders of magnitude more to fetch than a small, frequently accessed one [2606.20539.pdf, page 2]. For example, retaining a 1GB object accessed 10 times in a one-slot cache saves ∼$0.90 compared to retaining a 1KB object accessed 100 times, which saves only ∼$5 × 10⁻⁵, despite the latter achieving a higher hit rate [2606.20539.pdf, page 2]. This demonstrates that hit-rate optimization fails to account for the financial disparity between objects.

The financial suboptimality arises because cloud pric